# RR PRESSURE INDEX 2027 — SupeRR Selector Season 2
## Phase 01: Data Discovery, Forensic Audit & Participation Intelligence
**Consortium:** Senior IPL Auction Strategist, Lead Cricket Data Scientist, Senior Sports Statistician, Cricket Performance Analyst, Data Engineering Architect, Statistical Model Validation Specialist, Professional Power BI Analytics Consultant  
**Project Directory:** `RR_PRESSURE_INDEX_2027`  
**Evaluation Scope:** Senior Indian Premier League (IPL) and International Men's T20 (T20I) Competitions (2022–2026)


---
### 1. Environment Setup & Dependency Initialization
We initialize our analytical workspace using Python, Pandas, and NumPy with strict path independence.


In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Configure display options for franchise reporting
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# Resolve project base directory dynamically
NOTEBOOK_DIR = os.getcwd()
BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
AUDIT_DIR = os.path.join(BASE_DIR, "outputs", "audit")

print(f"Base Directory: {BASE_DIR}")
print(f"Raw Data Directory: {RAW_DIR}")
print(f"Audit Output Directory: {AUDIT_DIR}")


Base Directory: d:\RR_PRESSURE_INDEX_2027
Raw Data Directory: d:\RR_PRESSURE_INDEX_2027\data\raw
Audit Output Directory: d:\RR_PRESSURE_INDEX_2027\outputs\audit


---
### 2. Forensic Ingestion & Dataset Inventory
We ingest all six primary CSV files extracted into `data/raw/` and perform dimensional, memory, and null audits.


In [2]:
files = {
    "competitions": os.path.join(RAW_DIR, "competitions.csv"),
    "teams": os.path.join(RAW_DIR, "teams.csv"),
    "matches": os.path.join(RAW_DIR, "matches.csv"),
    "playing_eleven": os.path.join(RAW_DIR, "playing_eleven.csv"),
    "mapping": os.path.join(RAW_DIR, "competition_team_player_mapping.csv"),
    "ball_by_ball": os.path.join(RAW_DIR, "ball_by_ball.csv")
}

dfs = {}
for name, path in files.items():
    if name == "ball_by_ball":
        dfs[name] = pd.read_csv(path, low_memory=False)
    else:
        dfs[name] = pd.read_csv(path)
    print(f"Loaded {name:20s}: {dfs[name].shape[0]:>7,} rows | {dfs[name].shape[1]:>2} cols | {os.path.getsize(path)/(1024*1024):>6.2f} MB")


Loaded competitions        :     141 rows |  4 cols |   0.01 MB
Loaded teams               :     353 rows |  3 cols |   0.01 MB
Loaded matches             :     883 rows | 22 cols |   0.12 MB
Loaded playing_eleven      :  28,381 rows |  9 cols |   0.92 MB
Loaded mapping             :   6,413 rows |  6 cols |   0.44 MB
Loaded ball_by_ball        : 200,999 rows | 33 cols |  39.17 MB


---
### 3. Competition Scope Audit & Boundary Enforcement
The challenge mandate restricts analysis strictly to **IPL and T20I matches from 2022 onwards**.
We audit all competitions to separate senior eligible tournaments from developmental/domestic competitions.


In [3]:
comps = dfs["competitions"]
matches = dfs["matches"]

# Classify competitions
def classify_tournament(row):
    cname = str(row['comp_name'])
    mformat = str(row['match_format'])
    if 'IPL' in cname or 'Indian Premier League' in cname:
        return 'Included — IPL'
    elif mformat == 'T20I':
        return 'Included — T20I'
    else:
        return 'Excluded — Domestic/Development'

comps['scope_status'] = comps.apply(classify_tournament, axis=1)

print("Competition Breakdown by Scope Category:")
print(comps['scope_status'].value_counts())

print("\nExcluded Competitions:")
excluded_comps = comps[comps['scope_status'].str.startswith('Excluded')]
print(excluded_comps[['comp_id', 'comp_name', 'match_format', 'season']].to_string(index=False))


Competition Breakdown by Scope Category:
scope_status
Included — T20I                    134
Included — IPL                       5
Excluded — Domestic/Development      2
Name: count, dtype: int64

Excluded Competitions:
 comp_id                             comp_name match_format season
    2670 Mens T20 Emerging Teams Asia Cup 2024          T20   2024
    3289            Asia Cup Rising Stars 2025          T20   2025


In [4]:
# Merge matches with competition metadata
matches_scope = matches.merge(comps[['comp_id', 'comp_name', 'match_format', 'season', 'scope_status']], on='comp_id', how='left')

print("Match Breakdown across Scope Status:")
print(matches_scope['scope_status'].value_counts())

# Inspect the 16 abandoned matches with 0 balls bowled
bbb_match_ids = set(dfs['ball_by_ball']['match_id'].unique())
abandoned_matches = matches_scope[~matches_scope['match_id'].isin(bbb_match_ids)]
print(f"\nMatches in matches.csv with 0 deliveries in ball_by_ball.csv: {len(abandoned_matches)}")
print("Results of these 16 matches:")
print(abandoned_matches['result'].value_counts())


Match Breakdown across Scope Status:
scope_status
Included — T20I                    482
Included — IPL                     371
Excluded — Domestic/Development     30
Name: count, dtype: int64

Matches in matches.csv with 0 deliveries in ball_by_ball.csv: 16
Results of these 16 matches:
result
Match Abandoned    16
Name: count, dtype: int64


---
### 4. Forensic Investigation of Data Quality Anomalies
We investigate key variables in `ball_by_ball.csv`: `extras1_runs`, `runs`, `dismissal_type`, and `innings`.


In [5]:
bbb = dfs["ball_by_ball"]

# Anomaly 1: Decoding extras1_runs (Categorical Code vs Run Count)
print("Extras Type vs extras1_runs Cross-tabulation:")
print(pd.crosstab(bbb['extras_type'].fillna('None'), bbb['extras1_runs']))

print("\nForensic Conclusion: extras1_runs is NOT a run tally; it is an enum code mapping 1:1 with extras_type.")


Extras Type vs extras1_runs Cross-tabulation:
extras1_runs    1     2     3    4    5       6  7
extras_type                                       
None            0     0     0    0    0  189167  0
Overstep      576     0     0    0    0       0  0
Penalty         0     0     0    0    0       0  1
byes            0     0     0  529    0       0  0
legbyes         0     0  2748    0    0       0  0
other           0     0     0    0  264       0  0
wide            0  7714     0    0    0       0  0

Forensic Conclusion: extras1_runs is NOT a run tally; it is an enum code mapping 1:1 with extras_type.


In [6]:
# Anomaly 2: Striker Runs vs Extra Runs
print("Striker runs distribution across extras types:")
print(pd.crosstab(bbb['extras_type'].fillna('None'), bbb['runs']))

print("\nForensic Conclusion: 'runs' captures only runs off the striker's bat. Wides and no-balls require explicit penalty addition.")


Striker runs distribution across extras types:
runs             0      1      2    3      4   5      6
extras_type                                            
None         68029  72043  12496  653  23180  25  12741
Overstep       239    174     33    2     81   0     47
Penalty          1      0      0    0      0   0      0
byes           529      0      0    0      0   0      0
legbyes       2747      1      0    0      0   0      0
other          128     49     18    1     42   0     26
wide          7714      0      0    0      0   0      0

Forensic Conclusion: 'runs' captures only runs off the striker's bat. Wides and no-balls require explicit penalty addition.


In [7]:
# Anomaly 3: Dismissal Typo & Bowler Wicket Classification
print("Dismissal Types in ball_by_ball.csv:")
print(bbb['dismissal_type'].value_counts(dropna=False))

# Identify Obstructing typo
obstructing = bbb[bbb['dismissal_type'].str.contains('Obstr', case=False, na=False)]['dismissal_type'].value_counts()
print(f"\nObstructing field variants: {dict(obstructing)}")


Dismissal Types in ball_by_ball.csv:
dismissal_type
NaN                      190118
Caught                     7557
Bowled                     1775
LBW                         709
RunOut                      600
Stump                       214
Hit Wicket                   17
Retire Out                    6
Obstracting the field         2
Obstructing the field         1
Name: count, dtype: int64

Obstructing field variants: {'Obstracting the field': np.int64(2), 'Obstructing the field': np.int64(1)}


In [8]:
# Anomaly 4: Super Over Innings (Innings 3 & 4)
super_overs = bbb[bbb['innings'].isin([3, 4])]
print(f"Super Over Deliveries: {len(super_overs)} across {super_overs['match_id'].nunique()} matches.")
print("Super Over Match IDs:", super_overs['match_id'].unique())


Super Over Deliveries: 38 across 6 matches.
Super Over Match IDs: [34429 40725 45512 48027 48029 50359]


---
### 5. Player Participation & Canonical Master Profiling
Within the eligible scope (139 competitions, 853 matches, 194,014 deliveries), we profile all participating batters and bowlers.


In [9]:
# Load generated audit summary
player_summary_path = os.path.join(AUDIT_DIR, "player_participation_summary.csv")
df_players = pd.read_csv(player_summary_path)

print(f"Total Master Players in Eligible Scope: {len(df_players):,}")
print(f"Batters meeting sample threshold (>=100 balls): {df_players['qual_batter_100bf'].sum()}")
print(f"Bowlers meeting sample threshold (>=100 legal balls): {df_players['qual_bowler_100bb'].sum()}")
print(f"Dual All-rounders (>=100 each): {((df_players['qual_batter_100bf']) & (df_players['qual_bowler_100bb'])).sum()}")


Total Master Players in Eligible Scope: 1,132
Batters meeting sample threshold (>=100 balls): 303
Bowlers meeting sample threshold (>=100 legal balls): 311
Dual All-rounders (>=100 each): 108


In [10]:
print("Top 10 Aggregate Run Scorers (Eligible Scope 2022-2026):")
print(df_players[['player_id', 'player_name', 'total_matches', 'ipl_matches', 't20i_matches', 'balls_faced', 'runs_scored', 'batting_average', 'batting_strike_rate', 'sixes']].head(10).to_string(index=False))


Top 10 Aggregate Run Scorers (Eligible Scope 2022-2026):
 player_id          player_name  total_matches  ipl_matches  t20i_matches  balls_faced  runs_scored  batting_average  batting_strike_rate  sixes
      1699     Suryakumar Yadav            148           68            80         2761         4595            38.61               166.43    248
       813          Jos Buttler            131           75            56         2829         4396            41.08               155.39    204
      6281      Abhishek Sharma            134           71            63         2251         4198            33.32               186.49    275
      6288         Shubman Gill            118           79            39         2711         4050            40.50               149.39    142
        47          Virat Kohli             93           77            16         2456         3567            46.93               145.24    124
      4117         Ishan Kishan            127           73            54

In [11]:
print("Top 10 Aggregate Wicket Takers (Eligible Scope 2022-2026):")
df_bowlers = df_players.sort_values('wickets_taken', ascending=False)
print(df_bowlers[['player_id', 'player_name', 'total_matches', 'ipl_matches', 't20i_matches', 'overs_bowled', 'wickets_taken', 'bowling_average', 'economy_rate', 'bowling_strike_rate']].head(10).to_string(index=False))


Top 10 Aggregate Wicket Takers (Eligible Scope 2022-2026):
 player_id        player_name  total_matches  ipl_matches  t20i_matches  overs_bowled  wickets_taken  bowling_average  economy_rate  bowling_strike_rate
      6464     Arshdeep Singh            167           74            93         534.5            194            25.07          9.10                 16.5
      4722        Rashid Khan            126           79            47         457.8            153            22.16          7.40                 18.0
      6256 Varun Chakravarthy            113           69            44         354.7            135            21.01          8.00                 15.8
      4900  Wanindu Hasaranga            104           45            59         339.2            133            20.20          7.92                 15.3
      7503       Ravi Bishnoi            137           73            64         387.8            124            25.99          8.31                 18.8
      1612   Yuzvendra 

---
### 6. Analytical Feasibility & Context Feature Engineering
We verify the feasibility of deriving match pressure contexts:
- **Innings 2 Chase Pressure:** Target, Cumulative Runs, Balls Remaining, Wickets in Hand, Required Run Rate (RRR), CRR vs RRR leverage.
- **Innings 1 Par Pressure:** Par progression, collapse recovery (wickets in hand vs overs left), death overs acceleration leverage.


In [12]:
# Demonstrate delivery-level pressure context derivation on a sample match
sample_mid = 14094 # CSK vs KKR IPL 2022 Match 1
m_bbb = bbb[bbb['match_id'] == sample_mid].copy()

# Derived delivery metrics
m_bbb['is_legal'] = ~m_bbb['extras_type'].isin(['wide', 'Overstep', 'other'])
m_bbb['team_runs'] = m_bbb['runs'] + (m_bbb['extras_type'] == 'wide').astype(int) +                      (m_bbb['extras_type'].isin(['Overstep', 'other'])).astype(int) + m_bbb['extras_2_runs']

# Innings 2 Chase Reconstruction
chase_bbb = m_bbb[m_bbb['innings'] == 2].copy()
chase_bbb['cum_runs'] = chase_bbb['team_runs'].cumsum()
chase_bbb['cum_wickets'] = chase_bbb['is_wicket'].cumsum()
chase_bbb['wickets_remaining'] = 10 - chase_bbb['cum_wickets']
chase_bbb['legal_balls_bowled'] = chase_bbb['is_legal'].cumsum()
chase_bbb['balls_remaining'] = 120 - chase_bbb['legal_balls_bowled']

# Target: 132 (CSK scored 131)
target = 132
chase_bbb['runs_required'] = (target - chase_bbb['cum_runs']).clip(lower=0)
chase_bbb['rrr'] = np.where(chase_bbb['balls_remaining'] > 0, 
                            (chase_bbb['runs_required'] / chase_bbb['balls_remaining']) * 6.0, 0.0)

print("Sample Delivery Leverage Context (Match 14094, Chase Innings 2):")
print(chase_bbb[['overid', 'over_ball_num', 'striker_name', 'bowler_name', 'team_runs', 'cum_runs', 'wickets_remaining', 'balls_remaining', 'runs_required', 'rrr']].head(12).to_string(index=False))


Sample Delivery Leverage Context (Match 14094, Chase Innings 2):
 overid  over_ball_num   striker_name      bowler_name  team_runs  cum_runs  wickets_remaining  balls_remaining  runs_required      rrr
      1              1 Ajinkya Rahane Tushar Deshpande          0         0                 10              119            132 6.655462
      1              2 Ajinkya Rahane Tushar Deshpande          0         0                 10              118            132 6.711864
      1              3 Ajinkya Rahane Tushar Deshpande          0         0                 10              117            132 6.769231
      1              4 Ajinkya Rahane Tushar Deshpande          2         2                 10              116            130 6.724138
      1              5 Ajinkya Rahane Tushar Deshpande          4         6                 10              115            126 6.573913
      1              6 Ajinkya Rahane Tushar Deshpande          0         6                 10              114        

---
### 7. Phase 01 Audit Deliverables Verification
We confirm that all required audit deliverables have been generated and validated.


In [13]:
required_audit_files = [
    "dataset_inventory.csv",
    "data_quality_report.csv",
    "competition_scope_audit.csv",
    "player_participation_summary.csv"
]

print("Audit Deliverables Status:")
for fname in required_audit_files:
    fpath = os.path.join(AUDIT_DIR, fname)
    status = "EXISTS" if os.path.exists(fpath) else "MISSING"
    size_kb = os.path.getsize(fpath) / 1024 if os.path.exists(fpath) else 0
    print(f"  {fname:35s}: {status} ({size_kb:>7.2f} KB)")


Audit Deliverables Status:
  dataset_inventory.csv              : EXISTS (   1.18 KB)
  data_quality_report.csv            : EXISTS (   5.47 KB)
  competition_scope_audit.csv        : EXISTS (  23.22 KB)
  player_participation_summary.csv   : EXISTS ( 141.04 KB)
